## Spatial Distances

One of the flaw of previous distance analysis is that pmn-other cell distances were greatly influenced by the quanitity of other cells. If there are 100 PMNs and 4 CD4 cells than each CD4 will on average be matched with 25 PMN cells. This inflates distances in general. This new method limits the number of pairs to 1 per cell. So only 4 pairs are possible for the case above

### Load Protein Data

In [1]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np

# Load files on Evan's Laptop
# expr_orig = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\expression.csv", index_col=0)
# expr=expr_orig.transpose()
# metadata = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\metadata.csv", index_col=0)
# umap = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\umap.csv", index_col=0)

#Load files on Lab computer
expr_orig = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\expression.csv", index_col=0)
expr=expr_orig.transpose()
metadata = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\metadata.csv", index_col=0)
umap = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\umap.csv", index_col=0)

# Create AnnData object
adata = sc.AnnData(X=expr.values)

# Assign metadata
adata.obs = metadata
adata.var_names = expr.columns
adata.obs_names = expr.index

# Add spatial coordinates and UMAP to .obsm
# adata.obsm["spatial"] = metadata[['x_FOV_px', 'y_FOV_px']].values  # adjust if needed
adata.obsm["spatial"] = metadata[['x_FOV_px']].assign(y_FOV_px = -metadata['y_FOV_px']).values
adata.obsm["X_umap"] = umap.values

C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import annd

### Creating Distance Matrix

This creates a matrix of the distance of each pmn to each cell of another type.

It will have dimentions of (# PMN cell) * (# Target Cells). In cases of tumor cells or fibroblasts there can be many columns while for NK and Treg there will often be more pmn cells in a sample.

In [ ]:
import my_functions
import pandas as pd

sample_id="c_1_1"
target_cell="Macrophages"

# Gets Distance Matrix Between PMNs and Target Cells
results=my_functions.nearest_cells_of_particular_type(adata,sample_id,target_cell)


distance_matrix=results["Distance Matrix"]
distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)
display(distance_matrix)

num_pmns,num_target=distance_matrix.shape
display(num_pmns)
display(num_target)
#Create Lists of rows and columns
target_cell_list=distance_matrix.columns.tolist()
pmn_cell_list=distance_matrix.index.tolist()



In [32]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# --- 1. Define your Distance Matrix ---
# Let's assume you have 5 PMN cells (rows) and 3 macrophages (columns).
# The values represent the distance between each PMN and macrophage.
distance_matrix = np.array([
    [82, 83, 69],  # PMN 1 to Macrophages 1, 2, 3
    [77, 37, 49],  # PMN 2 to Macrophages 1, 2, 3
    [11, 69,  5],  # PMN 3 to Macrophages 1, 2, 3
    [74, 92, 35],  # PMN 4 to Macrophages 1, 2, 3
    [  8, 9, 98]   # PMN 5 to Macrophages 1, 2, 3
])

# --- 2. Apply the Hungarian Algorithm ---
# The linear_sum_assignment function will find the optimal assignment
# that minimizes the sum of the distances.
pmn_indices, macrophage_indices = linear_sum_assignment(distance_matrix)

# --- 3. Extract the Results ---
# The function returns the optimal row (PMN) and column (macrophage) indices.
optimal_pairs = list(zip(pmn_indices, macrophage_indices))

# Calculate the minimum total distance
min_total_distance = distance_matrix[pmn_indices, macrophage_indices].sum()

# --- 4. Display the Optimal Pairings and Total Distance ---
print("Optimal PMN-Macrophage Pairs (PMN index, Macrophage index):")
for pmn_idx, mac_idx in optimal_pairs:
    distance = distance_matrix[pmn_idx, mac_idx]
    print(f"  PMN {pmn_idx} is paired with Macrophage {mac_idx} (Distance: {distance})")

print(f"\nMinimum Overall Distance: {min_total_distance}")

# Note on unpaired PMNs:
all_pmn_indices = set(range(distance_matrix.shape[0]))
paired_pmn_indices = set(pmn_indices)
unpaired_pmn_indices = all_pmn_indices - paired_pmn_indices

print("\nUnpaired PMN cells (by index):")
print(f"  {list(unpaired_pmn_indices)}")

Optimal PMN-Macrophage Pairs (PMN index, Macrophage index):
  PMN 1 is paired with Macrophage 1 (Distance: 37)
  PMN 2 is paired with Macrophage 2 (Distance: 5)
  PMN 4 is paired with Macrophage 0 (Distance: 8)

Minimum Overall Distance: 50

Unpaired PMN cells (by index):
  [0, 3]


In [67]:
import my_functions
def mininum_distance_exclusive_pairing(distance_matrix):
    import pandas as pd
    import numpy as np
    from scipy.optimize import linear_sum_assignment
    
    # --- 1. Define your Distance Matrix as a Pandas DataFrame ---
    distance_data=distance_matrix
    distance_df = pd.DataFrame(distance_data)
    
    # --- 2. Apply the Hungarian Algorithm ---
    # The scipy function requires a NumPy array, so we extract it using .to_numpy()
    cost_matrix = distance_df.to_numpy()
    pmn_indices, target_indices = linear_sum_assignment(cost_matrix)
    
    # --- 3. Map Indices back to DataFrame Labels ---
    # Get the actual labels from the DataFrame's index and columns
    paired_pmns = distance_df.index[pmn_indices]
    paired_targets = distance_df.columns[target_indices]
    
    # Get the distances for the optimal pairs
    optimal_distances = cost_matrix[pmn_indices, target_indices]
    
    # --- 4. Create a DataFrame for the Results and Display ---
    # This provides a clean, readable output of the optimal pairings.
    results_df = pd.DataFrame({
        'PMN_Cell': paired_pmns,
        'Paired_Target': paired_targets,
        'Distance': optimal_distances
    })
    
    average_distance = results_df['Distance'].mean()
    print(f'The average distance is: {average_distance}')
    
    # Calculate the minimum total distance
    min_total_distance = optimal_distances.sum()
    print(f"\nMinimum Overall Distance: {min_total_distance}")
    
    # Identify unpaired PMNs using the DataFrame's index
    all_pmns = set(distance_df.index)
    paired_pmns_set = set(paired_pmns)
    unpaired_pmns = all_pmns - paired_pmns_set

    return{
        "Pair Matrix":results_df,
        "Average Distance":average_distance
    }

sample_id="c_1_1"
target_cell="CD4+T_cells"

# Gets Distance Matrix Between PMNs and Target Cells
results=my_functions.nearest_cells_of_particular_type(adata,sample_id,target_cell)
distance_matrix=results["Distance Matrix"]
distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)

results=mininum_distance_exclusive_pairing(distance_matrix)
display(results["Pair Matrix"])
print(f'The average distance is: {results["Average Distance"]}')

The average distance is: 436.8167220452529

Minimum Overall Distance: 3931.350498407276


,PMN_Cell,Paired_Target,Distance
0,c_1_1_384,c_1_1_365,436.854306
1,c_1_1_675,c_1_1_600,670.396711
2,c_1_1_942,c_1_1_5127,418.373972
3,c_1_1_1880,c_1_1_1775,304.399543
4,c_1_1_1908,c_1_1_1904,555.072229
5,c_1_1_2008,c_1_1_61,385.760925
6,c_1_1_2109,c_1_1_75,577.585197
7,c_1_1_2286,c_1_1_2332,576.617142
8,c_1_1_6363,c_1_1_6412,6.290473


The average distance is: 436.8167220452529
